In [1]:
import os
# Disable GPU usage (GPUs add no value for the small example and we do not need to fight for ressources this way) 
os.environ["CUDA_VISIBLE_DEVICES"]="-1"
import tensorflow as tf

# Gesten erkennen mit Neuronalen Netzwerken

Wir wollen noch einmal den Gestendatensatz aus Übung Nummer 6 verwenden. Diesmal wollen wir diesen aber mit einem neuronalen Netzwerk klassifizieren.

Zuerst müssen wir die Daten laden. Die Tensorflow Methode [image_dataset_from_directory](https://www.tensorflow.org/api_docs/python/tf/keras/utils/image_dataset_from_directory) erledigt das für uns, unter der Annahme, dass die Bilder in eigenen Verzeichnissen je Klasse liegen (was bei uns der Fall ist). Nötige Parameter:
- `directory`: Das Verzeichnis, in dem sich die Unterverzeichnisse je Klasse befinden
- `shuffle`: Wollen wir die Bilder zufällig permutieren (Ja, ansonsten kommen ja z.B. erst alle 0en, dann alle 1en....)
- `batch_size`: Wir trainieren mit minibatch stochastic gradient descent (Mittelweg zwischen nur einem Beispiel je Schritt und allen Beispielen). Hier geben wir die Größe eines Minibatches an (64 passst ganz gut).
- `image_size`: Ein Tupel mit der Zielgröße, in die die Bilder skaliert werden sollen. Wir wollen 40x40 nehmen.
- `validation_split`: Welchen Anteil wollen wir als Validation-Set behalten (20%)
- `subset`: Entweder "training" oder "validation". Gibt an, welches der beiden Teil Sets zurückgegeben werden soll.
- `seed`: Random Seed für Shuffle. Muss der gleiche für das Trainings- und Validation-Set sein, damit diese sicher disjunkt sind.

Lesen Sie nun ein Trainings- und ein Validation-Set ein.

In [2]:
import tensorflow as tf

train_dataset = tf.keras.utils.image_dataset_from_directory(
    directory = "../UB6/Dataset",
    shuffle = True,
    batch_size = 64,
    image_size = (40,40),
    validation_split = 0.2,
    subset = "training",
    seed = 1
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    directory = "../UB6/Dataset",
    shuffle = True,
    batch_size = 64,
    image_size = (40,40),
    validation_split = 0.2,
    subset = "validation",
    seed = 1
)

import tensorflow as tf

print(tf.__version__)
print(tf.keras.__version__)

Found 2071 files belonging to 10 classes.
Using 1657 files for training.
Found 2071 files belonging to 10 classes.
Using 414 files for validation.
2.20.0
3.12.0


Ein Dataset verhält sich ungefähr wie ein Generator. Man kann darüber iterieren (ohne dass alle Daten notwendigerweise im Speicher liegen müssen).
Wir können mit `element_spec` schauen, welche Form unsere Daten haben.

In [3]:
train_dataset.element_spec

(TensorSpec(shape=(None, 40, 40, 3), dtype=tf.float32, name=None),
 TensorSpec(shape=(None,), dtype=tf.int32, name=None))

In [4]:
for x in train_dataset:
    print(x)
    break

(<tf.Tensor: shape=(64, 40, 40, 3), dtype=float32, numpy=
array([[[[129.4375, 125.4375, 122.4375],
         [133.8125, 129.8125, 126.8125],
         [138.8125, 134.8125, 131.8125],
         ...,
         [129.5   , 124.5   , 121.5   ],
         [125.125 , 120.875 , 119.375 ],
         [121.5   , 117.5   , 116.5   ]],

        [[134.1875, 130.1875, 127.1875],
         [138.5625, 134.5625, 131.5625],
         [144.1875, 140.1875, 137.1875],
         ...,
         [133.75  , 128.75  , 125.75  ],
         [129.5   , 125.25  , 123.75  ],
         [125.8125, 121.8125, 120.8125]],

        [[138.8125, 134.8125, 131.8125],
         [144.1875, 140.1875, 137.1875],
         [151.5   , 147.5   , 144.5   ],
         ...,
         [138.1875, 133.9375, 132.4375],
         [133.0625, 129.    , 127.875 ],
         [129.0625, 125.0625, 124.0625]],

        ...,

        [[142.375 , 139.375 , 132.375 ],
         [148.75  , 145.75  , 138.75  ],
         [154.1875, 151.1875, 144.1875],
         ...,
     

## Logistische Regression

Als erstes wollen wir einmal logistische Regression mit Tensorflow machen.
Mit `tf.keras.models.Sequential` können wir ein Modell erstellen, das aus mehreren Layern besteht (das können wir dann später einfach um mehr ebenen Erweitern). Diesem geben wir eine Liste mit Layern mit.

- `tf.keras.layers.Rescaling`: Ist ein Layer, um unsere Input-Features zu skalieren (wieder sollten wir durch 255 teilen, damit die Werte zwischen 0 und 1 liegen. Probieren Sie mal aus, was passiert, wenn wir das weglassen.
- `tf.keras.layers.Flatten`: Ist ein Layer, der die 40x40x3 Bilder in einen langen Vektor überführt.
- `tf.keras.layers.Dense` Ist ein regulärer linearer Layer, so wie wir ihn kennen. Diesem geben wir mit
    - `units`: Wieviele Outputs soll der Layer haben (wir haben 10 Klassen, also brauchen wir auch so viele Outputs)
    - `activation`: Welche Aktivierungsfunktion soll verwendet werden (für unseren Fall ist "softmax" die richtige)
    - `kernel_regularizer`: Wie soll regularisiert werden. Hier können wir L2 (`tf.keras.regularizers.L2`) Regularisierung mit einem Gewicht von 0.01 nehmen (Sie können gerne etwas experimentieren, was hier gut funktioniert.)
    
Mit dem definierten Modell müssen wir nun spezifizieren, wie es optimiert werden soll. Dafür rufen wir `model.compile` auf. Diesem müssen wir mitgeben:
- `optimizer`: Welcher Optimierungsalgorithmus verwendet werden soll. `tf.optimizers.SGD` ist normaler Stochastic Gradient Descent (auf Minibatches von der Größe, wie sie vom Datensatz geliefert werden). Wir müssen hier eine Lernrate angeben. Probieren Sie aus, was gut funktioniert.
- `loss`: Welche Zielfunktion soll optimiert werden. Für Klassifikation ist "CategoricalCrossentropy" Oft die richtige Wahl. Hier (und in vielen anderen Fällen) liegen die Klassen als Integer (0, 1, 2, ..., 9) vor. Mit der Loss Funktion `tf.losses.SparseCategoricalCrossentropy` brauchen wir nicht selber ein One-Hot Encoding der Klassen zu machen, sondern Tensorflow weiß, dass es das intern machen soll.

Ein weiterer Parameter, den wir angeben können, ist:
- `metrics`: Eine Liste mit zusätzlichen Metriken, die getrackt werden sollen. Hier können wir `tf.metrics.SparseCategoricalAccuracy` angeben, um die Accuracy direkt mit ausgedruckt zu bekommen.

Als letztes müssen wir das Modell trainieren. Das tun wir mit `model.fit`. Hier geben wir mit:
- Das train_dataset
- `epochs`: Die Anzahl an vollständigen Durchläufen durch das gesamte Trainingsset. Mit circa 200-300 kann man zu brauchbaren Ergebnissen kommen.

Optional können wir direkt noch mit angeben:
- `validation_data`: Wenn wir ein Validation Set mit angeben, dann bekommen wir nach jeder Epoche auch alle Metriken für die Validation Daten berechnet.

In [ ]:
model = tf.keras.models.Sequential([
    # ADD CODE HERE
    tf.keras.layers.Rescaling(1./255), #Beim teilen durch 255 erhalten wir kleinere float werte. Wenn wir das nicht machen, wird den Ausgabewert vom Neuron immer größer, was dazu führt, dass der Schritt von Gradient-Update zu hoch ist. (по слайдам перепроверить себя)
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(units=10, #jedes neuron bekommt 784 features und 1 bias. Bias(b) ist intercept. allgemein für logistische regression: z=w1x1+w2x2+wnxn+b -> activation function -> output
                           activation= "softmax",
                             kernel_regularizer= tf.keras.regularizers.L2(0.01) #model kriegt penalty für sehr hohe weights, deshalb werden sie gleichmäßiger verteilt (quadratische Regularisierung)
    )
    # dense layer definiert meherer lineare modelle, die parallel arbeiten
    # wenn wir meherer dense layers hinzufügen, wächst das modell in tiefe
    # wenn wir den parameter units ändert, wächst das modell in breite
])

model.compile( 
    optimizer = tf.optimizers.SGD(1e-2),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics = [tf.metrics.SparseCategoricalAccuracy]
)
# ADD CODE HERE

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:
epochs = 300

history = model.fit(
    train_dataset,
    validation_data = validation_dataset,
    epochs = epochs
)

Epoch 1/300
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 3.2563 - sparse_categorical_accuracy: 0.1225 - val_loss: 3.0986 - val_sparse_categorical_accuracy: 0.1401
Epoch 2/300
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 3.1641 - sparse_categorical_accuracy: 0.1557 - val_loss: 2.6067 - val_sparse_categorical_accuracy: 0.1570
Epoch 3/300
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 2.7993 - sparse_categorical_accuracy: 0.2064 - val_loss: 3.6224 - val_sparse_categorical_accuracy: 0.1232
Epoch 4/300
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.6883 - sparse_categorical_accuracy: 0.2438 - val_loss: 2.6990 - val_sparse_categorical_accuracy: 0.2681
Epoch 5/300
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2.4611 - sparse_categorical_accuracy: 0.2800 - val_loss: 2.5773 - val_sparse_categorical_accuracy: 0.3357
Epoch 6/300
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2.4133 - sparse_categorical_accuracy: 0.3084 - val_loss: 1.9876 - val_sparse_categorical_accuracy: 0.3792
Epoch 7/30

## Neuronales Netzwerk

Ein neuronales Netzwerk ist jetzt leicht definiert. Wir fügen einfach mehr Layer hinzu. Fügen Sie zum Beispiel nach dem "Flatten" Layer noch zwei Layer mit 50 bzw. 25 Outputs und "relu" Activations hinzu.

Hier sieht man schnell, was Neuronale Netze schwierig in der Handhabung macht. Auf einmal haben wir eine sehr große Anzahl an Hyperparametern, die eingestellt werden müssen.

Beim Versuch das Netzwerk zu trainieren, fällt Ihnen eventuell auf, dass das Training am Anfang guten Fortschritt macht, während dann später der Loss immer öfter Sprünge nach oben macht. Je besser wir werden, umso mehr macht eine kleine Lernrate Sinn, damit wir den Fortschritt nicht mehr verlieren.
Mit `tf.keras.optimizers.schedules.ExponentialDecay` können wir eine sinkende Lernrate implementieren, die wir dann "SGD" als Parameter mitgeben. Hier können wir angeben:
- `initial_learning_rate`: Die Lernrate im ersten Schritt (im ersten Mini Batch)
- `decay_steps`: Nach wievielen Minibatches soll die Lernrate jeweils verringert werden
- `decay_rate`: Um welchen Faktor soll die Lernrate jeweils verringert werden (typischerweise >= 0.9 und < 1.0).

Mit 3 Layern kommen wir auf ein Ergebnis, dass über dem aus der Cross Validation-Übung liegt. Schaffen Sie es, auch besser als die SVM zu werden? Ich jedenfalls nicht ;-)

In [ ]:
# ADD CODE HERE
model = tf.keras.models.Sequential([
    # ADD CODE HERE
    tf.keras.layers.Rescaling(1./255),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(units=50,
                           activation= "relu",
                           )
    ,
    tf.keras.layers.Dense(units=25,
                           activation= "relu",
                          )
    ,
    tf.keras.layers.Dense(units=10,
                           activation= "softmax",
                           ),
])

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate= 1e-2,
    decay_steps= 1000, #270
    decay_rate=0.95 #0.97
)

model.compile(
    optimizer = tf.keras.optimizers.SGD(lr_schedule),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics = [tf.metrics.SparseCategoricalAccuracy()]
)

Mit `model.summary()` können Sie sich eine Zusammenfassung des Modells drucken. Unter anderem können wir hier direkt sehen, dass unser Modell fast eine Viertelmillion Parameter hat.

In [11]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_2 (Rescaling)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Sie müssen mit dem Training eines Modells nicht bei 0 anfangen. Indem Sie erneut `.fit` aufrufen, können Sie das Training eines Modell einfach mit ein paar zusätzlichen Epochen fortsetzen. Dabei können Sie `initial_epoch` angeben, damit die Funktion weiß, wo sie aufgehört hat (wichtig für das Scaling der Lernrate). Die Anzahl Epochen vom `epoch` Parameter ist dabei nicht die Anzahl an Extra Epochen, sondern wieviele Sie ingesamt trainieren möchten.

In [12]:
epochs = 300

history = model.fit(
    train_dataset,
    validation_data = validation_dataset,
    epochs = epochs
)

Epoch 1/300
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 2.3105 - sparse_categorical_accuracy: 0.0960 - val_loss: 2.2936 - val_sparse_categorical_accuracy: 0.1111
Epoch 2/300
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2.2961 - sparse_categorical_accuracy: 0.1183 - val_loss: 2.2924 - val_sparse_categorical_accuracy: 0.0990
Epoch 3/300
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.2851 - sparse_categorical_accuracy: 0.1291 - val_loss: 2.2775 - val_sparse_categorical_accuracy: 0.1329
Epoch 4/300
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2.2659 - sparse_categorical_accuracy: 0.1605 - val_loss: 2.2735 - val_sparse_categorical_accuracy: 0.1908
Epoch 5/300
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.2317 - sparse_categorical_accuracy: 0.1690 - val_loss: 2.2149 - val_sparse_categorical_accuracy: 0.2077
Epoch 6/300
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2.2201 - sparse_categorical_accuracy: 0.1931 - val_loss: 2.1898 - val_sparse_categorical_accuracy: 0.1473
Epoch 7/3

## Additionaly: Manuelles Fit